# Harness를 활용한 자동 시각적 QA

| 정보 | 세부 내용 |
|---|---|
| 튜토리얼 | 에이전트 microVM 안에서 웹 앱 빌드, 실행 및 시각적 테스트 수행 |
| SDK | boto3 |
| 모델 | Claude Haiku 4.5 (Bedrock) |
| 기능 | ExecuteCommand, 사용자 지정 컨테이너, 헤드리스 브라우저(Puppeteer) |

### 핵심 아이디어

웹 앱을 AI 에이전트에 전달하고 *"build it, run it, test it, and show me screenshots of every step."*이라고 요청할 수 있다고 가정해 보겠습니다. 이 노트북에서 바로 그 작업을 수행합니다.

Harness microVM은 자체 파일 시스템과 네트워크 스택을 갖춘 완전한 Linux 환경입니다. 에이전트는 격리된 환경에서 도구를 설치하고 서버를 시작하며 헤드리스 브라우저를 실행할 수 있습니다. 따라서 **자동 시각적 QA**에 적합합니다.

- **CI/CD 파이프라인** - 커밋할 때마다 에이전트가 앱을 실행하고 시각적 테스트를 수행하여 코드 리뷰 전에 UI 회귀를 표시합니다.
- **버전 간 비교** - 두 버전의 앱을 나란히 빌드하고 각각 스크린샷을 찍어 차이를 비교합니다.
- **탐색적 QA** - 에이전트에 URL과 *"find anything that looks broken"*이라는 요청을 주면 에이전트가 탐색하고 상호 작용한 뒤 결과를 보고합니다.
- **온보딩 문서** - 에이전트가 앱을 단계별로 실행하며 주석이 포함된 스크린샷 안내서를 자동으로 생성합니다.

### 구축할 항목

간단하지만 완전한 예제를 구축합니다. 에이전트가 TodoMVC 앱을 생성하여 `localhost:3000`에서 제공하고, Puppeteer(헤드리스 Chrome)를 설치한 뒤 항목 추가, 완료 표시, 스크린샷 촬영을 수행하는 테스트 스크립트를 작성하고 실행합니다. 이후 스크린샷을 노트북으로 가져옵니다.

**핵심:** Puppeteer는 웹 서버와 같은 VM 안에서 실행되므로 네트워크 격리 문제 없이 `localhost`를 사용할 수 있습니다.

## Part 0: 설정

In [ ]:
import sys
import time
import uuid
from pathlib import Path
import boto3

# 헬퍼
sys.path.insert(0, str(Path.cwd().parent.parent))

# --- 설정 ---
from helper.iam import create_harness_role, delete_harness_role
from helper.client import get_agentcore_control_client, get_agentcore_client

# --- boto3 클라이언트 생성 ---
control = get_agentcore_control_client()
client = get_agentcore_client()

account_id = boto3.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}")

In [ ]:
role_arn = create_harness_role()
print(f"\nExecution Role ARN: {role_arn}")

print("Waiting for IAM role to propagate...")
time.sleep(10)
print("Ready!")

## Part 1: Node.js 컨테이너를 사용하는 Harness 생성

에이전트가 웹 서버와 Puppeteer를 모두 실행하려면 Node.js가 필요합니다. Harness에 `node:20-slim` 컨테이너를 연결하며, 이 컨테이너가 microVM의 기본 OS가 됩니다.

In [ ]:
HARNESS_NAME = f"WebAppTester_{uuid.uuid4().hex[:8]}"

resp = control.create_harness(
    harnessName=HARNESS_NAME,
    executionRoleArn=role_arn,
)
harness = resp["harness"]
harness_id = harness["harnessId"]
harness_arn = harness["arn"]
print(f"Harness ID: {harness_id}")
print(f"Harness ARN: {harness_arn}")

In [ ]:
control.update_harness(
    harnessId=harness_id,
    environmentArtifact={
        "optionalValue": {"containerConfiguration": {"containerUri": "public.ecr.aws/docker/library/node:20-slim"}}
    },
)
print("Attaching Node.js container...")

for i in range(24):
    resp = control.get_harness(harnessId=harness_id)
    status = resp["harness"]["status"]
    print(f"  {status}")
    if status == "READY":
        print("\u2705 Harness is ready")
        break
    time.sleep(5)

## Part 2: 환경 준비

Harness microVM의 장점을 활용하는 단계입니다. `ExecuteCommand`를 사용하여 다음 작업을 수행합니다.
1. 시스템 종속성 설치(git, curl, Chromium)
2. 웹 앱 클론 또는 생성
3. `localhost:3000`에서 앱 제공
4. 헤드리스 브라우저 자동화를 위한 `puppeteer-core` 설치

모든 작업은 격리된 VM 안에서 이루어지며 로컬 시스템에는 영향을 주지 않습니다.

In [ ]:
session_id = str(uuid.uuid4()).upper()


def run_command(cmd):
    """에이전트 microVM에서 명령을 실행하고 stdout을 반환합니다."""
    output = ""
    resp = client.invoke_agent_runtime_command(
        agentRuntimeArn=harness_arn, runtimeSessionId=session_id, body={"command": cmd}
    )
    for event in resp["stream"]:
        if "chunk" in event and "contentDelta" in event["chunk"]:
            delta = event["chunk"]["contentDelta"]
            if "stdout" in delta:
                output += delta["stdout"]
            if "stderr" in delta:
                output += delta["stderr"]
    return output


print("Installing git and dependencies...")
out = run_command("apt-get update -qq && apt-get install -y -qq git curl chromium > /dev/null 2>&1 && echo 'done'")
print(out.strip())

print("\nCloning TodoMVC...")
out = run_command("git clone --depth 1 https://github.com/tastejs/todomvc.git /tmp/todomvc 2>&1")
print(out[-300:] if len(out) > 300 else out)

In [ ]:
# 자체 완결형 TodoMVC 앱 생성(저장소 종속성 문제 방지)
print("Creating TodoMVC app...")
run_command("mkdir -p /tmp/todomvc")

# 에이전트에 앱 생성을 요청
response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "Create a single-file TodoMVC app at /tmp/todomvc/index.html. It should be a complete, self-contained HTML file with inline CSS and JS. Features: add todos, toggle complete, filter (All/Active/Completed), delete. Use a clean modern design. No external dependencies."
                }
            ],
        }
    ],
)
for event in response["stream"]:
    if "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()

# 확인
out = run_command("ls -la /tmp/todomvc/index.html")
print(f"\n{out.strip()}")

In [ ]:
# 서버 시작
print("Starting server...")
run_command("cd /tmp/todomvc && nohup npx -y serve -l 3000 > /tmp/server.log 2>&1 &")
time.sleep(5)

out = run_command("curl -s -o /dev/null -w '%{http_code}' http://localhost:3000")
print(f"Server status: {out.strip()}")

# Puppeteer 설치
print("\nInstalling Puppeteer (this takes ~1 min)...")
out = run_command("cd /tmp && npm install puppeteer-core 2>&1 | tail -3")
print(out)
print("\u2705 Ready")

## Part 3: 에이전트가 테스트 작성 및 실행

가장 강력한 단계입니다. 테스트 스크립트를 직접 작성하는 대신 **에이전트에 작성을 요청**합니다. 에이전트는 다음 작업을 수행합니다.

1. 방금 구축한 앱 구조 파악
2. 자연어로 설명한 테스트 단계가 포함된 Puppeteer 스크립트 작성
3. VM 안에서 스크립트 실행
4. 각 단계의 스크린샷 저장

실제 QA 파이프라인에서는 TodoMVC 앱을 자체 앱으로 대체할 수 있습니다. 저장소를 클론하고 빌드한 다음 에이전트가 탐색하게 합니다. 패턴은 동일합니다.

```
ExecuteCommand: git clone + npm install + npm start
ExecuteCommand: npm install puppeteer-core
invoke_harness: "Write and run tests for the app at localhost:3000"
ExecuteCommand: base64 /tmp/screenshot_*.png → 가져오기
```

In [ ]:
# test_session = str(uuid.uuid4()).upper()
print(f"Test session: {session_id}\n")


response = client.invoke_harness(
    harnessArn=harness_arn,
    runtimeSessionId=session_id,
    model={"bedrockModelConfig": {"modelId": "global.anthropic.claude-haiku-4-5-20251001-v1:0"}},
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": """There is a TodoMVC web app running at http://localhost:3000 and puppeteer-core is installed at /tmp/node_modules/puppeteer-core. Chromium is at /usr/bin/chromium.

Write a Puppeteer test script at /tmp/test.mjs and run it. The script should:

1. Launch chromium (headless, no-sandbox) and open http://localhost:3000
2. Take screenshot → /tmp/screenshot_1.png (empty app)
3. Add three todos: 'Book flights to Amsterdam', 'Reserve hotel', 'Plan museum visits'
4. Take screenshot → /tmp/screenshot_2.png (three todos)
5. Click the checkbox on 'Book flights to Amsterdam' to mark it complete
6. Take screenshot → /tmp/screenshot_3.png (one completed)
7. Close the browser

Use import from '/tmp/node_modules/puppeteer-core/lib/esm/puppeteer/puppeteer-core.js' or require('/tmp/node_modules/puppeteer-core').
After writing the script, run it with: node /tmp/test.mjs
Then list the screenshots: ls -la /tmp/screenshot_*.png"""
                }
            ],
        }
    ],
)

for event in response["stream"]:
    if "contentBlockStart" in event:
        start = event["contentBlockStart"].get("start", {})
        if "toolUse" in start:
            print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
    elif "contentBlockDelta" in event:
        delta = event["contentBlockDelta"].get("delta", {})
        if "text" in delta:
            print(delta["text"], end="", flush=True)
    elif "messageStop" in event:
        print()

## Part 4: 스크린샷 가져오기 및 검토

스크린샷은 에이전트 VM에 저장됩니다. `ExecuteCommand`를 통해 base64로 인코딩된 데이터를 가져와 인라인으로 표시합니다.

프로덕션 파이프라인에서는 다음과 같이 활용할 수 있습니다.
- 보관을 위해 S3에 업로드
- 기준 스크린샷과 비교(시각적 회귀)
- PR에 댓글로 첨부
- 에이전트에 다시 전달: *"Do these screenshots look correct?"*

In [ ]:
import base64
from IPython.display import Image, display

out = run_command("ls -la /tmp/screenshot_*.png 2>/dev/null || echo 'No screenshots found'")
print(out)

screenshots = []
for i in range(1, 10):
    b64 = ""
    resp = client.invoke_agent_runtime_command(
        agentRuntimeArn=harness_arn,
        runtimeSessionId=session_id,
        body={"command": f"base64 /tmp/screenshot_{i}.png 2>/dev/null"},
    )
    for event in resp["stream"]:
        if "chunk" in event and "contentDelta" in event["chunk"]:
            delta = event["chunk"]["contentDelta"]
            if "stdout" in delta:
                b64 += delta["stdout"]

    if not b64.strip():
        break

    b64_clean = b64.strip().replace("\n", "").replace("\r", "").replace(" ", "")
    remainder = len(b64_clean) % 4
    if remainder:
        b64_clean = b64_clean[:-remainder]

    try:
        img_bytes = base64.b64decode(b64_clean)
        screenshots.append(img_bytes)
        print(f"\nScreenshot {i}: {len(img_bytes):,} bytes")
        display(Image(data=img_bytes))
    except Exception as e:
        print(f"\nScreenshot {i}: falha no decode — {e}")

print(f"\n\u2705 Retrieved {len(screenshots)} screenshots")

## 리소스 정리

In [ ]:
control.delete_harness(harnessId=harness_id)
print(f"Deleted harness: {harness_id}")

In [ ]:
delete_harness_role()